# Configuration

In [ ]:
import os
import pandas as p
import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback
)
from datasets import load_dataset, ClassLabel, Value, Dataset, DatasetDict
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training
)
from trl import SFTTrainer
import gc
import numpy as np

# Set Global Variables

In [ ]:
# Model Specific
model_id = "PATH/TO/MODEL"
model_name = "MODEL_NAME"

# QLORA Specific
r = 8
lora_alpha = 8

#Training Specific
learning_rate = 1e-07
batch_size = 16
output_dir = f'PATH/TO/OUTPUT/DIRECTORY'

# Datasets

In [ ]:
#Both data files should be formatted with a "prompt" and "completion" column both of which are str
training_data = pd.read_csv("PATH/TO/TRAINING/DATA")
validation_data = pd.read_csv("DPATH/TO/VALIDATION/DATA")

training_dataset = Dataset.from_pandas(training_df)
validation_dataset = Dataset.from_pandas(validation_df)
dataset = DatasetDict({"train":training_dataset, "test": validation_dataset})
#dataset = dataset.rename_columns({'0' : 'prompt', 'scores' : 'completion'}) #If columns are improperly named make sure to correct

# Functions and Classes

In [ ]:
class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.learning_rate = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if "loss" in logs:
            self.train_losses.append(logs["loss"])
        if "eval_loss" in logs:
            self.eval_losses.append(logs["eval_loss"])
        if "learning_rate" in logs:
            self.learning_rate.append(logs["learning_rate"])

# Model & Tokenizer Loading

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant = True
)

model = AutoModelForCausalLM.from_pretrained(
        model_id,
        low_cpu_mem_usage=True,
        torch_dtype=torch.bfloat16,
        quantization_config = bnb_config,
        device_map = 'auto'
    )
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# QLORA

In [ ]:
peft_config = LoraConfig(
        lora_alpha = lora_alpha,
        r = r,
        bias = "none",
        task_type = "CAUSAL_LM",
        target_modules =  ['q_proj', 'v_proj', 'k_proj', 'o_proj']
    )
model = get_peft_model(model, peft_config)

# Training

In [ ]:
torch.cuda.empty_cache()
gc.collect()

loss_logger = LossLoggerCallback()

training_args = TrainingArguments(
        num_train_epochs = 100,
        learning_rate = learning_rate,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size = batch_size,
        gradient_accumulation_steps = 1,
        logging_strategy = "epoch",
        save_strategy= "epoch",
        output_dir = output_dir,
        logging_dir = f"{output_dir}/logs",
        optim = "adamw_torch",
        warmup_ratio = 0.1,
        do_eval = True,
        eval_strategy = "epoch",
        gradient_checkpointing = True,
        load_best_model_at_end = True        
    )

sft_trainer = SFTTrainer(
        model,
        args = training_args,
        train_dataset = dataset['train'],
        eval_dataset = dataset['test'],
        processing_class = tokenizer,
        callbacks = [loss_logger, EarlyStoppingCallback(early_stopping_patience = 3, early_stopping_threshold = 0.1)]
    )

In [ ]:
# Train
print("Training Model...")
torch.cuda.empty_cache()
sft_trainer.train()

# Save Model
output_dir = os.path.join(output_dir, "final_checkpoint")
print("Saving Pretrained Model...")
sft_trainer.model.save_pretrained(output_dir)
print("Saving Pretrained Tokenizer...")
tokenizer.save_pretrained(output_dir)